## I will start by reading & Understanding the data

In [1]:
import pandas as pd
df = pd.read_csv(r'..\data\LoanData_Bondora.csv')

C:\Users\User\AppData\Local\Temp\ipykernel_6236\2876369551.py:2: DtypeWarning: Columns (34,38,80,82,83,89,90) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r'..\data\LoanData_Bondora.csv')


In [27]:
df.duplicated().sum()

np.int64(0)

See the columns, identify the target:

In [28]:
df.columns

Index(['ReportAsOfEOD', 'LoanId', 'LoanNumber', 'ListedOnUTC',
       'BiddingStartedOn', 'BidsPortfolioManager', 'BidsApi', 'BidsManual',
       'UserName', 'NewCreditCustomer',
       ...
       'PreviousEarlyRepaymentsCountBeforeLoan', 'GracePeriodStart',
       'GracePeriodEnd', 'NextPaymentDate', 'NextPaymentNr',
       'NrOfScheduledPayments', 'ReScheduledOn', 'PrincipalDebtServicingCost',
       'InterestAndPenaltyDebtServicingCost', 'ActiveLateLastPaymentCategory'],
      dtype='object', length=112)

'Status' shows us some individuals are lay to pay, some pay fully, and some loans have been declared in default (What we want to predict).

In [29]:
df['Status'].value_counts()

Status
Late       68574
Current    57774
Repaid     52887
Name: count, dtype: int64

In [30]:
df['Default'] = df['Status'].isin(['Late', 'Default']).astype(int)
df = df.drop(columns=['Status'])

This dataset inherently includes predictions + Post-decision details that won't be present at inference + Other non-predictive columns which I will drop.

In [ ]:
cols_to_drop = [
    # IDs / metadata
    'ReportAsOfEOD', 'LoanId', 'LoanNumber', 'UserName', 'ListedOnUTC',
    'BiddingStartedOn', 'LoanApplicationStartedDate', 'LanguageCode',
    'ModelVersion', 'City', 'County',

    # P2P marketplace mechanics
    'BidsPortfolioManager', 'BidsApi', 'BidsManual',

    # post-origination leakage
    'DefaultDate', 'LoanDate', 'ContractEndDate', 'FirstPaymentDate',
    'MaturityDate_Original', 'MaturityDate_Last',
    'PrincipalPaymentsMade', 'InterestAndPenaltyPaymentsMade',
    'PrincipalWriteOffs', 'InterestAndPenaltyWriteOffs',
    'PrincipalBalance', 'InterestAndPenaltyBalance',
    'PrincipalOverdueBySchedule', 'PlannedPrincipalPostDefault',
    'PlannedInterestPostDefault', 'PlannedPrincipalTillDate', 'PlannedInterestTillDate',
    'CurrentDebtDaysPrimary', 'CurrentDebtDaysSecondary', 'EAD1', 'EAD2',
    'PrincipalRecovery', 'InterestRecovery', 'RecoveryStage',
    'GracePeriodStart', 'GracePeriodEnd', 'NextPaymentDate', 'NextPaymentNr',
    'NrOfScheduledPayments', 'LastPaymentOn', 'ReScheduledOn', 'StageActiveSince',
    'ActiveLateCategory', 'WorseLateCategory', 'ActiveLateLastPaymentCategory',
    'ActiveScheduleFirstPaymentReached', 'Restructured',
    'PrincipalDebtServicingCost', 'InterestAndPenaltyDebtServicingCost',
    'DebtOccuredOn', 'DebtOccuredOnForSecondary',

    # Bondora's own model outputs
    'Rating', 'Rating_V0', 'Rating_V1', 'Rating_V2',
    'ExpectedLoss', 'LossGivenDefault', 'ExpectedReturn', 'ProbabilityOfDefault',
    'EL_V0', 'EL_V1', 'CreditScoreEeMini', 'CreditScoreEsMicroL',
    'CreditScoreEsEquifaxRisk', 'CreditScoreFiAsiakasTietoRiskGrade', 'Amount', 'Interest',

    # Redundant
    'DateOfBirth',  # keeping Age instead
]
# NOTE: previous version was missing a comma after 'Interest', which silently
# concatenated it with 'DateOfBirth' into one bad string -> neither column
# actually got dropped. Fixed here.

df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
print(df.shape)
print(list(df.columns))

In [32]:
df.isna().sum()

NewCreditCustomer                              0
ApplicationSignedHour                          0
ApplicationSignedWeekday                       0
VerificationType                              50
Age                                            0
DateOfBirth                               179235
Gender                                        45
Country                                        0
AppliedAmount                                  0
Interest                                       0
LoanDuration                                   0
MonthlyPayment                              6690
UseOfLoan                                      0
Education                                     50
MaritalStatus                                 50
NrOfDependants                            143635
EmploymentStatus                             202
EmploymentDurationCurrentEmployer           1606
EmploymentPosition                        179235
WorkExperience                            142713
OccupationArea      

We can start with checking our categorical columns:

In [33]:
cat_cols = df.select_dtypes(exclude='number').columns

print("\nCATEGORICAL COLUMNS:")
print(list(cat_cols))


CATEGORICAL COLUMNS:
['NewCreditCustomer', 'Country', 'NrOfDependants', 'EmploymentDurationCurrentEmployer', 'WorkExperience']


In [34]:
df['NewCreditCustomer'] = df['NewCreditCustomer'].astype(int)

In [35]:
df['Country'].value_counts() 

Country
EE    110714
FI     41955
ES     26270
SK       296
Name: count, dtype: int64

In [36]:
df['NrOfDependants'].value_counts()

NrOfDependants
0         15624
1          6086
0.0        4421
2          3647
1.0        2254
2.0        1383
3          1151
3.0         446
4           300
4.0         122
5            72
5.0          46
6            19
7            11
10Plus        6
7.0           4
6.0           4
8.0           2
10            2
Name: count, dtype: int64

In [37]:
def clean_dependants(x):
    if pd.isna(x):
        return 'Unknown'
    
    # Handle "10Plus"
    if str(x) == '10Plus':
        return '10'
    
    # Convert floats/strings like "2.0" -> "2"
    return str(min(int(float(x)), 10))

df['NrOfDependants'] = df['NrOfDependants'].apply(clean_dependants) # In our full system, we can enforce int data input.

In [38]:
df['NrOfDependants'].value_counts()

NrOfDependants
Unknown    143635
0           20045
1            8340
2            5030
3            1597
4             422
5             118
6              23
7              15
10              8
8               2
Name: count, dtype: int64

The following features have a hierarchical nature (increasing levels of work experience) which means a numeric encoding serves us well. For special cases (like, Retiree) we dedicate columns. 

In [39]:
df['EmploymentDurationCurrentEmployer'].value_counts()

EmploymentDurationCurrentEmployer
MoreThan5Years    67139
UpTo5Years        42615
UpTo1Year         32899
Retiree           10286
Other              8301
UpTo2Years         6504
UpTo3Years         5444
UpTo4Years         3690
TrialPeriod         751
Name: count, dtype: int64

In [40]:
employment_map = {
    'TrialPeriod': 0,
    'UpTo1Year': 1,
    'UpTo2Years': 2,
    'UpTo3Years': 3,
    'UpTo4Years': 4,
    'UpTo5Years': 5,
    'MoreThan5Years': 6,
    'Retiree': 7,
    'Other': -1
}

df['employment_duration_ord'] = (
    df['EmploymentDurationCurrentEmployer']
    .map(employment_map)
)


In [41]:
df['WorkExperience'].value_counts()

WorkExperience
15To25Years        7902
5To10Years         7834
10To15Years        6900
MoreThan25Years    6675
2To5Years          5122
LessThan2Years     2089
Name: count, dtype: int64

In [42]:
work_exp_map = {
    'LessThan2Years': 0,
    '2To5Years': 1,
    '5To10Years': 2,
    '10To15Years': 3,
    '15To25Years': 4,
    'MoreThan25Years': 5
}

df['work_experience_ord'] = df['WorkExperience'].map(work_exp_map)

For the most part, we have resolved / decided how to act with each of our categorical features. Now, numeric:

In [43]:
num_cols = df.select_dtypes(include='number').columns

print("NUMERIC COLUMNS:")
print(list(num_cols))

NUMERIC COLUMNS:
['NewCreditCustomer', 'ApplicationSignedHour', 'ApplicationSignedWeekday', 'VerificationType', 'Age', 'DateOfBirth', 'Gender', 'AppliedAmount', 'Interest', 'LoanDuration', 'MonthlyPayment', 'UseOfLoan', 'Education', 'MaritalStatus', 'EmploymentStatus', 'EmploymentPosition', 'OccupationArea', 'HomeOwnershipType', 'IncomeFromPrincipalEmployer', 'IncomeFromPension', 'IncomeFromFamilyAllowance', 'IncomeFromSocialWelfare', 'IncomeFromLeavePay', 'IncomeFromChildSupport', 'IncomeOther', 'IncomeTotal', 'ExistingLiabilities', 'LiabilitiesTotal', 'RefinanceLiabilities', 'DebtToIncome', 'FreeCash', 'MonthlyPaymentDay', 'NoOfPreviousLoansBeforeLoan', 'AmountOfPreviousLoansBeforeLoan', 'PreviousRepaymentsBeforeLoan', 'PreviousEarlyRepaymentsBefoleLoan', 'PreviousEarlyRepaymentsCountBeforeLoan', 'Default', 'employment_duration_ord', 'work_experience_ord']


In [44]:
df['ApplicationSignedHour'].value_counts()

import numpy as np

# Convert hour column to numeric just in case
df['ApplicationSignedHour'] = pd.to_numeric(
    df['ApplicationSignedHour'],
    errors='coerce'
)

# Cyclical encoding
df['ApplicationSignedHour_sin'] = np.sin(
    2 * np.pi * df['ApplicationSignedHour'] / 24
)

df['ApplicationSignedHour_cos'] = np.cos(
    2 * np.pi * df['ApplicationSignedHour'] / 24
)

# Optional: drop original column
df.drop(columns=['ApplicationSignedHour'], inplace=True)

In [45]:
df['ApplicationSignedHour_cos'].isna().sum()

np.int64(0)

In [46]:
df['Gender_Male'] = (df['Gender'] == 0.0).astype(int)
df['Gender_Female'] = (df['Gender'] == 1.0).astype(int)
df['Gender_Unknown'] = (df['Gender'] == 2.0).astype(int)
df.drop(columns=['Gender'], inplace=True)

In [49]:
df['UseOfLoan'].value_counts()

UseOfLoan
-1      142647
 7        9700
 2        9191
 0        6914
 6        3151
 3        1952
 5        1787
 8        1519
 4        1366
 1         955
 102        21
 110        17
 104         6
 101         5
 107         2
 106         1
 108         1
Name: count, dtype: int64

Discrete values confirms its categorical, and -1 is unknown. 

In [51]:
income_columns = df.columns[df.columns.str.contains('Income')]

### MonthlyPayment and MonthlyPaymentDay

`MonthlyPayment` is computed from `Amount` and `Interest` (already dropped as post-decision leakage), so it inherits the same problem and gets dropped too. `MonthlyPaymentDay` is an administrative scheduling detail assigned after origination, not an applicant input or a risk signal — drop it as well.

In [ ]:
df = df.drop(columns=[c for c in ['MonthlyPayment', 'MonthlyPaymentDay'] if c in df.columns])
print(df.shape)

### Previous loan history — NaN likely means "first loan", not missing

`PreviousRepaymentsBeforeLoan` and `PreviousEarlyRepaymentsBefoleLoan` have a lot of NaNs. Check whether these line up with first-time borrowers (`NoOfPreviousLoansBeforeLoan == 0`) before deciding how to fill them.

In [ ]:
# Sanity check: do the NaNs correspond to first-time borrowers?
print(df.loc[df['PreviousRepaymentsBeforeLoan'].isna(), 'NoOfPreviousLoansBeforeLoan'].describe())

In [ ]:
# If confirmed above (mostly 0 previous loans), fillna with 0 rather than median -
# median would falsely imply they had some repayment history.
prev_loan_cols = ['PreviousRepaymentsBeforeLoan', 'PreviousEarlyRepaymentsBefoleLoan',
                   'PreviousEarlyRepaymentsCountBeforeLoan']
df[prev_loan_cols] = df[prev_loan_cols].fillna(0)
df['NoOfPreviousLoansBeforeLoan'] = df['NoOfPreviousLoansBeforeLoan'].fillna(0)
df['AmountOfPreviousLoansBeforeLoan'] = df['AmountOfPreviousLoansBeforeLoan'].fillna(0)

### Income breakdown columns

Check how sparse the per-source income columns are before deciding whether to engineer composition features or just drop them in favor of `IncomeTotal`.

In [ ]:
income_cols = ['IncomeFromPrincipalEmployer', 'IncomeFromPension',
               'IncomeFromFamilyAllowance', 'IncomeFromSocialWelfare',
               'IncomeFromLeavePay', 'IncomeFromChildSupport', 'IncomeOther']

for col in income_cols:
    print(col, (df[col] > 0).mean())

mask_all_zero = (df[income_cols] == 0).all(axis=1)
print('\n%% rows with all components 0:', mask_all_zero.mean())
print(df.loc[mask_all_zero, 'IncomeTotal'].describe())

Income source breakdown is too sparse/unreliable to trust (most "all-zero" rows still have a positive `IncomeTotal`, meaning the breakdown wasn't consistently populated across the dataset's history). Drop the raw breakdown, keep `IncomeTotal`, and flag rows where it was reported as exactly 0 (likely missing/unreported rather than truly zero income) before imputing.

In [ ]:
# Median computed only from genuinely reported (non-zero) incomes
income_median = df.loc[df['IncomeTotal'] > 0, 'IncomeTotal'].median()

df['income_unknown'] = (df['IncomeTotal'] == 0).astype(int)
df['IncomeTotal'] = df['IncomeTotal'].replace(0, income_median)

df = df.drop(columns=income_cols)
print(df.shape)

### Education — ordinal

Check the actual codes before mapping (same lesson learned from Gender — don't trust a remembered mapping).

In [ ]:
df['Education'].value_counts()
# TODO: confirm exact code->level mapping against Bondora docs before mapping ordinally
# (expected: Primary < Basic < Vocational < Secondary < Higher, with -1 = not set)

### Remaining nominal categoricals -> one-hot

No natural order between categories (e.g. "Self-employed" isn't "more" than "Retiree"), low-to-moderate cardinality each.

In [ ]:
one_hot_cols = [
    'VerificationType', 'MaritalStatus', 'EmploymentStatus',
    'HomeOwnershipType', 'Country',
]

df[one_hot_cols].nunique()

### High-cardinality nominal -> WoE

`UseOfLoan` and `OccupationArea` have too many categories for clean one-hot without exploding dimensionality. Weight of Evidence per Section 3.3 of the report.

In [ ]:
woe_cols = ['UseOfLoan', 'OccupationArea']
df[woe_cols].nunique()
# WoE encoding to be applied against `Default` once final feature set is locked in

### ApplicationSignedWeekday — apply the same cyclical treatment as the hour

A linear/distance-based model would otherwise see Sunday and Monday as maximally far apart, when in reality the week wraps around.

In [ ]:
df['ApplicationSignedWeekday_sin'] = np.sin(2 * np.pi * df['ApplicationSignedWeekday'] / 7)
df['ApplicationSignedWeekday_cos'] = np.cos(2 * np.pi * df['ApplicationSignedWeekday'] / 7)
df.drop(columns=['ApplicationSignedWeekday'], inplace=True)

In [55]:
# Check sparsity first - how common is each income source
income_cols = ['IncomeFromPrincipalEmployer', 'IncomeFromPension',
               'IncomeFromFamilyAllowance', 'IncomeFromSocialWelfare',
               'IncomeFromLeavePay', 'IncomeFromChildSupport', 'IncomeOther']

for col in income_cols:
    print(col, "            ", (df[col] > 0).mean())  # % of applicants with non-zero income from this source

IncomeFromPrincipalEmployer              0.19119033670879013
IncomeFromPension              0.022054844198956678
IncomeFromFamilyAllowance              0.03297347058331241
IncomeFromSocialWelfare              0.010120791140123302
IncomeFromLeavePay              0.004329511535135437
IncomeFromChildSupport              0.007286523279493403
IncomeOther              0.03081987335062906


In [ ]:
# Engineer composition signal, then drop the raw breakdown
df['EmploymentIncomeRatio'] = df['IncomeFromPrincipalEmployer'] / df['IncomeTotal'].replace(0, pd.NA)
df['NonEmploymentIncomeFlag'] = (
    (df['IncomeFromPension'] > 0) |
    (df['IncomeFromSocialWelfare'] > 0) |
    (df['IncomeFromFamilyAllowance'] > 0) |
    (df['IncomeFromChildSupport'] > 0)
).astype(int)
df['IncomeSourceCount'] = (df[income_cols] > 0).sum(axis=1)

df = df.drop(columns=income_cols)  # keep IncomeTotal, drop the raw breakdown

In [57]:
df.shape

(179235, 47)